# 01 - Exploratory Data Analysis (EDA)
### Customer Churn Prediction in the Banking Sector

**Dataset:** [Credit Card customers - Kaggle](https://www.kaggle.com/datasets/anwarsan/credit-card-bank-churn)

This notebook covers:
1. Loading the raw dataset
2. Basic info, shape, dtypes, missing/unknown values
3. Target variable (`Attrition_Flag`) distribution (churn vs non-churn)
4. Distribution plots for quantitative features (to decide scaling method later)
5. Boxplots to spot outliers / narrow-range features
6. Categorical feature distributions
7. Correlation heatmap

Goal: understand the data well enough to design the preprocessing pipeline (notebook 02).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Load the Dataset

In [ ]:
DATA_PATH = "../data/raw/BankChurners.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


According to the paper, the original dataset has **23 columns**. The **last two columns**
(Naive Bayes Classifier helper columns auto-generated by Kaggle) are not useful for prediction
and should be dropped, leaving **21 variables** and **10,127 rows**.


In [ ]:
# Drop the last two Naive-Bayes helper columns (as done in the paper)
cols_to_drop = [c for c in df.columns if 'Naive_Bayes' in c]
print("Dropping columns:", cols_to_drop)

df = df.drop(columns=cols_to_drop)

# Drop CLIENTNUM later during preprocessing (kept here for EDA reference)
print("Shape after dropping helper columns:", df.shape)
df.head()


## 2. Basic Info, Dtypes, Missing / Unknown Values

In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


In [ ]:
# Standard missing-value check
print("Missing values per column:")
print(df.isnull().sum())


In [ ]:
# The dataset often encodes missing data as the string 'Unknown' instead of NaN
# (common in Education_Level, Marital_Status, Income_Category)
unknown_counts = (df == 'Unknown').sum()
unknown_counts = unknown_counts[unknown_counts > 0]
print("Columns containing 'Unknown' values:")
print(unknown_counts)


In [ ]:
# Visualize proportion of 'Unknown' per affected column
if len(unknown_counts) > 0:
    (unknown_counts / len(df) * 100).plot(kind='bar', color='salmon')
    plt.ylabel('% Unknown')
    plt.title("Percentage of 'Unknown' values per column")
    plt.show()


## 3. Target Variable: `Attrition_Flag`

- `Existing Customer` = stayed (non-churn)
- `Attrited Customer` = churned

This is an **imbalanced classification problem** — most customers stay, only a minority churn.
This is exactly why the paper applies **SMOTE** later in the pipeline.


In [ ]:
target_counts = df['Attrition_Flag'].value_counts()
target_pct = df['Attrition_Flag'].value_counts(normalize=True) * 100

print(target_counts)
print(target_pct.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Attrition_Flag', data=df, ax=ax[0], palette='Set2')
ax[0].set_title('Churn vs Non-Churn (count)')

ax[1].pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
          colors=sns.color_palette('Set2'), startangle=90)
ax[1].set_title('Churn vs Non-Churn (%)')

plt.tight_layout()
plt.show()


**Observation:** The dataset is imbalanced (~16% churn vs ~84% non-churn, ratio roughly 1:5.4
as reported in the paper). This imbalance must be addressed before training models
(handled with SMOTE in notebook 03/04).


## 4. Quantitative Feature Distributions

We separate the quantitative columns and plot histograms to see which ones look
**normally distributed** (candidates for **standardization**) vs **skewed / wide-range**
(candidates for **normalization**) — this mirrors Table 2 in the paper.


In [ ]:
quant_cols = [
    'Customer_Age', 'Months_on_book', 'Total_Relationship_Count',
    'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit',
    'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1',
    'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1',
    'Avg_Utilization_Ratio', 'Dependent_count'
]

quant_cols = [c for c in quant_cols if c in df.columns]
df[quant_cols].hist(bins=30, figsize=(18, 14), color='steelblue', edgecolor='black')
plt.suptitle("Distribution of Quantitative Features", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()


**Observation:**
- `Customer_Age` and `Months_on_book` look roughly **normally distributed** → standardization.
- `Credit_Limit`, `Avg_Open_To_Buy`, `Total_Revolving_Bal`, `Total_Trans_Amt`, `Total_Trans_Ct`
  have **wide/skewed ranges** → normalization.
- `Dependent_count`, `Total_Relationship_Count`, `Months_Inactive_12_mon`,
  `Contacts_Count_12_mon`, `Total_Amt_Chng_Q4_Q1`, `Total_Ct_Chng_Q4_Q1`,
  `Avg_Utilization_Ratio` have a **narrow range of variance** → leave unchanged.

(This matches Table 2 of the reference paper.)


## 5. Boxplots — Outlier Detection on Narrow-Range Features

In [ ]:
narrow_range_cols = [
    'Dependent_count', 'Total_Relationship_Count', 'Months_Inactive_12_mon',
    'Contacts_Count_12_mon', 'Total_Amt_Chng_Q4_Q1', 'Total_Ct_Chng_Q4_Q1',
    'Avg_Utilization_Ratio'
]
narrow_range_cols = [c for c in narrow_range_cols if c in df.columns]

plt.figure(figsize=(10, 8))
sns.boxplot(data=df[narrow_range_cols], orient='h', palette='Set3')
plt.title("Quantitative Data Chart (Narrow-Range Features)")
plt.xlabel("Values")
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots for the wide-range / skewed features
wide_range_cols = ['Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy',
                    'Total_Trans_Amt', 'Total_Trans_Ct']
wide_range_cols = [c for c in wide_range_cols if c in df.columns]

fig, axes = plt.subplots(1, len(wide_range_cols), figsize=(20, 5))
for ax, col in zip(axes, wide_range_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightblue')
    ax.set_title(col)
plt.tight_layout()
plt.show()


## 6. Categorical Feature Distributions

In [ ]:
cat_cols = ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']
cat_cols = [c for c in cat_cols if c in df.columns]

fig, axes = plt.subplots(len(cat_cols), 1, figsize=(10, 4 * len(cat_cols)))
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(y=col, data=df, order=order, ax=ax, palette='viridis')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()


In [ ]:
# Churn rate broken down by each categorical feature
for col in cat_cols:
    churn_rate = pd.crosstab(df[col], df['Attrition_Flag'], normalize='index') * 100
    print(f"\nChurn rate (%) by {col}:")
    print(churn_rate.round(2))


In [ ]:
fig, axes = plt.subplots(len(cat_cols), 1, figsize=(10, 4 * len(cat_cols)))
for ax, col in zip(axes, cat_cols):
    pd.crosstab(df[col], df['Attrition_Flag'], normalize='index').plot(
        kind='bar', stacked=True, ax=ax, colormap='Set2'
    )
    ax.set_title(f'Churn Proportion by {col}')
    ax.set_ylabel('Proportion')
    ax.legend(title='Attrition_Flag', loc='upper right')
plt.tight_layout()
plt.show()


## 7. Correlation Heatmap (Numeric Features)

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])

plt.figure(figsize=(14, 10))
corr = numeric_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title("Correlation Heatmap of Numeric Features")
plt.tight_layout()
plt.show()


In [ ]:
# Top correlated pairs (excluding self-correlation)
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0]
print("Top 10 most correlated feature pairs:")
print(corr_pairs.head(10))


**Observation:** Features like `Credit_Limit` and `Avg_Open_To_Buy` are likely highly
correlated (since Avg_Open_To_Buy = Credit_Limit - Total_Revolving_Bal). This is useful
to know for feature selection / multicollinearity checks later.


## 8. EDA Summary & Next Steps

**Key takeaways:**
- Target is imbalanced (~84% stay vs ~16% churn) → need SMOTE.
- `Unknown` values exist in some categorical columns → need cleaning.
- `Customer_Age` & `Months_on_book` ≈ normal distribution → standardize.
- Several quantitative features are wide-range/skewed → normalize.
- A few quantitative features have narrow variance → leave unchanged.
- Categorical features have 2 values (Gender) or many values (Education_Level,
  Marital_Status, Income_Category, Card_Category) → label encoding vs one-hot encoding
  respectively.
- `Credit_Limit` and `Avg_Open_To_Buy` are strongly correlated — worth keeping in mind.

➡️ **Next:** `02_preprocessing.ipynb` — clean unknowns, encode categoricals, scale
quantitative features, and prepare the final modeling dataset.
